# Estimation of the Runoff coefficient of European river catchments

<div class="alert alert-block alert-info">
    <b>Structure of the Test</b>
    <p>The test contains two types of questions:</p>
    <ul>
        <li><b><em>Questions AX</em></b> on <b>coding</b> require practical coding or analysis of results.</li>
        <li><b><em>Questions CX</em></b> on the <b>course</b> focus on theoretical concepts and can be treated independently of the coding tasks.</li>
        <li><b><em>Questions BX</em></b> are 2 <b>bonus coding</b> questions providing extra points if answered correctly.</li>
    </ul>
</div>

<div class="alert alert-block alert-warning">
    <p>
        <b>Instructions:</b>
        <ul>
            <li><b>The use of AI chatbots such as ChatGPT is forbidden.</b></li>
            <li>The input data file (CSV file <code>flood_data_europe.csv</code>) must be in the same directory as the notebook.</li>
        </ul>
    </p>
</div>


## Introduction

In October 2024, a massive flood occurred in Valencia, Spain, when over 445 mm of rain fell in just 24 hours, killing more than 200 people. The magnitude of this event can be explained in part by the **runoff coefficient (RC)**. The Runoff Coefficient is a parameter between 0 and 1 that measures what fraction of rainfall becomes surface runoff rather than infiltrating into the ground. Valencia's extensive urbanization had created vast impermeable surfaces with high runoff coefficients (0.7-0.95), meaning most of the torrential rainfall immediately became destructive surface flow instead of being absorbed by natural vegetation, which would have had much lower runoff coefficients (0.1-0.4). 

**Accurately characterizing and anticipating RC is crucial in flood forecasting**.

In this notebook, we intend to analyze the role of land/meteorological variables in modulating RC. We will compare the ability of different variables (e.g., earlier discharge, surface soil moisture at different depths, snow water content and anetecedent precipitation) to explain observed variability of RC across historical European floods by using regression.

### Dataset presentation


- The dataset in the file `flood_data_europe.csv` consists of 3061 flood events that took place in one of the 102 European basins. For each flood event, hydrological information is provided to characterize it, along with metadata such as  the latitude and longitude of the basins.

- The short names and the descriptions of the variables are provided below.

> **Discharge information**
>- `Rc`: Runoff Coefficient (no unit)

> **Hydrological information** (measure at basin scale; the given value is the average over the 3day prior to the flood event): 
>- `SM_ERA1`: soil moisture in the 1st layer of soil (0-7 cm) (unit: m$^3$/m$^3$)
>- `SM_ERA2`: soil moisture in the 2nd layer of soil (7-28 cm) (unit: m$^3$/m$^3$)
>- `SM_ERA3`: soil moisture in the 3rd layer of soil (28-100 cm) (unit: m$^3$/m$^3$)
>- `API5`:  antecedent precipitation index (unit: mm)
>- `preQ`: Logarithm of pre storm river discharge (unit: log(mm))
>- `SWE`: snow water equivalent before the event (unit: mm)

  Source: ECMWF Reanalysis [(ERA5 data base](https://cds.climate.copernicus.eu/cdsapp#!/dataset/reanalysis-era5-pressure-levels))
  
> **Type of basins:**
> - `class`: a categorical variable defining the type of each basin. `class = 0` : Non Mediteranean basin; `class = 1`: Mediterean basin.
> - `basin_id`: an identifier (integer) to label each basin individually.  

> **Input metadata on the basin:**
> - `lat`: latitude (degree north)
> - `lon`: longitude (degree east)

### Code preparation

The following code cell is provided in order to read the database and is not intended to be modified. We advise you to keep the variable `df_all` unchanged and create new variables when working with a subset.

In [1]:
from pathlib import Path
import pandas as pd

# Define the path to the database
data_dir = Path('') # Fill this in case the .csv file is not located in the same directory as this notebook
Basin_filename = 'flood_data_europe.csv'
Basin_filepath = Path(data_dir, Basin_filename)

# Read the database                                                                                                                
df_all = pd.read_csv(Basin_filepath)

## Dataset sampling and training strategy

As you can see the 3061 flood event are happening on to 142 different bassins that can be classified into two classes : Mediterean and Non-Mediterranean.

    
> ***Question A1***
> - In this dataset, what is the total number of data points $N$, and the maximum number of hydrological input features $p$ available for prediction $RC$?

In [6]:
df_all.shape

(3061, 11)

In [3]:
df_all.describe()
#Based on .describe() the number of data points N is 3061*11 points.The number of freatures to describe RC would be 10 = 11-1 since we are not considering the target variable RC. 
#The number of features to describe RC is 10 since we are not considering the target variable RC.

,Rc,preQ,API5,SWE,SM_ERA1,SM_ERA2,SM_ERA3,lat,lon,class,basin_id
count,3061.000000,3061.000000,3061.000000,3061.000000,3061.000000,3061.000000,3061.000000,3061.000000,3061.000000,3061.000000,3061.000000
mean,0.209118,3.816494,6.456645,43.061732,1.663075,0.348972,0.336911,45.416972,2.344103,0.467821,77.683437
std,0.182209,3.167980,7.328413,383.066351,2.271279,0.068620,0.076171,4.531210,9.241352,0.499045,45.147566
min,0.000004,-11.512839,0.000047,0.000000,0.051207,0.032000,0.046000,37.785556,-9.156969,0.000000,0.000000
25%,0.066910,2.016242,1.414974,0.000000,0.436919,0.319875,0.296000,41.644000,-5.933889,0.000000,36.000000
50%,0.167048,3.655579,3.987331,0.000000,0.833371,0.367333,0.362200,44.903011,-0.315000,0.000000,81.000000
75%,0.305494,5.300851,8.650297,0.000000,1.904080,0.394222,0.388000,48.606248,7.898373,1.000000,116.000000
max,0.997387,23.985047,58.039512,8398.822000,20.262786,0.590000,0.553000,63.986207,27.512222,1.000000,148.000000


In [7]:
df_all.head()

,Rc,preQ,API5,SWE,SM_ERA1,SM_ERA2,SM_ERA3,lat,lon,class,basin_id
0,0.445676,6.182155,2.798431,2.1364,3.357808,0.3884,0.3968,49.307181,4.782227,0,0
1,0.249433,2.934733,0.572006,0.0000,1.284453,0.3562,0.3750,49.307181,4.782227,0,0
2,0.141071,2.878732,4.811485,0.0000,0.632515,0.3334,0.2930,49.307181,4.782227,0,0
3,0.187594,4.620551,11.781655,0.0000,0.932474,0.4108,0.4034,49.307181,4.782227,0,0
4,0.160699,4.088665,7.864114,0.0000,0.771563,0.3908,0.3906,49.307181,4.782227,0,0


Answer


> ***Question A2***
> - Do you think this dataset is composed of iid samples? Justify your answer.

A2 asks whether each row can be treated as an **iid sample**:

- **Independent:** knowing about one flood event gives no information about another.
- **Identically distributed:** all events come from the same underlying statistical distribution.

Think about whether events from the same basin—or events with similar basin characteristics—could be related or have systematically different runoff behavior. The question asks you to use that context to judge whether the iid assumption is reasonable.

Firstly, IDD samples stand for samples that are independent and identically distributed. Hence, if the basins are signifcally distant, I would assume they´re independent. However, I wouldn´t be able to tell whether suppossing that the samples are identically distributed would be a strong assumption or not.

Answer



> ***Question A3***
> - Propose two training strategies regarding independance sampling?

Answer



## Basin selection: the regional dataset

In the code below we select flood events that occured near Valencia. We will refer to this subset as the **regional dataset**.

> ***Question A4***
> - Comment and execute the code below. 

In [2]:
import numpy as np
lat_valencia = 39.4699075
lon_valencia = -0.3762881
dist = np.sqrt((df_all['lat'] - lat_valencia)**2 + (df_all['lon'] - lon_valencia)**2)
id = np.argsort(dist)
id2 = list(df_all.iloc[id[:]]['basin_id'])
id3 = sorted(set(id2), key=id2.index)
n_basins = 10
df_sub = df_all[df_all['basin_id'].isin(id3[:n_basins])]

> ***Question A5***
> - Are all basins well representated in the dataset? You can answer either with a plot or with numbers. Explain the plot or numbers with one or two sentences.

In [3]:
# your answer here


## Estimating RC based on hydrological information.

Let us first use only the pre storm river discharge (`preQ`) to **linearly estimate** the runoff coefficient (`Rc`).
 
> ***Question A6***
> - Select these variables in the shuffled regional dataset. Split the input and output into a training (80% of the dataset) and testing (20% of the dataset).

In [4]:
# Your code here


> ***Question A7***
> - Use the train set to do a linear regression using `preQ` to predict `Rc`.

In [5]:
# Your code here


> ***Question A8***
> - For this dataset, plot `preQ` as a function of `Rc` along with your linear regression. Use different colors for the train and test sets. Don't forget the labels and the legend.

In [6]:
# Your code here


> ***Question A9***
> - What are the train and test score?

In [7]:
# Your code here


> ***Question A10***
> - Re-execute at least 5 times the random train/test split and the linear fit. What comments can you do on both the magnitude and variability of the train and test scores?

Answer:



> ***Question A11***
> - If you had the possibility to either add more feature or add more data, which option would you choose and why?

Answer: 

In the rest of the test, we will try these two options and see which one performs best

## Option 1: More features for the regional subset

We still consider the **regional** dataset but we now keep all features **except** `lat`, `lon`, `class`, and `basin_id`

> ***Question A12***
> - Prepare a train and a test set (with random shuffling of the regional dataset).
> - Train and test a linear model for this new dataset. Compute a $R^2$ score for the training and testing set

In [8]:
# Your code here


We want to apply the **LASSO** regularization technique to select the most important features. LASSO is similar to the ridge technique except that the regularization term is the sum of the absolute value of the coefficients instead of the norm squared.
 
> ***Question A13***
> - What is an important preliminary step before applying Ridge or LASSO regularisation method? Why?

Answer:


> ***Question A14***
> - Do this step.

In [9]:
# your code here


We work with a regularization parameter $\alpha$ and we do not not know in advance the best range of variation of $\alpha$. 

> ***Question A15***
> - create a `np.array` with 100 values of alphas using `np.logspace`. Choose arbitrary bounds, this will be refined later.

In [10]:
# your code here


The goal here is to write a regularization loop. We are going to adopt the following strategy: for each value of the regularization parameter $\alpha$ we shuffle the regional dataset and split it into a training and testing set, train the model on the training set and test it on the testing set. For each value of $\alpha$, we work with a different training set.

> ***Question A16***
> - Explain why this is a form of **nested** cross validation.

Answer:


> ***Question A17***
> - Do a Lasso regularization and store (i) the value of the coefficients of the regression, (ii) $R^2$ for the training set, and (iii) $R^2$ for the test set. Perform a different train/test split for each value of $\alpha$ (method described above).

In [11]:
# your code here


> ***Question A18***
> - Plot $R^2$ for the training set and for the testing set as a function of $\alpha$. Based on what you obtain, you will need to adjust the limits of your interval of $\alpha$ (and re-execute your code above).

In [12]:
# your code here


> ***Question A19***
> - What do you conclude?

Answer:

> ***Question A20***
> - Plot the values of the regression coefficients as a function of $\alpha$. Do not forget the labels so that we can identify each curve.

In [13]:
# your code here


> ***Question A21***
> - What is the most important coefficient? Explain

Your text answer here:


## Option 2: More samples with only one input feature

We go back to the problem with only one input feature and try to predict `RC`. We now consider all the flood events (we do not restrict ourselves to the regional dataset). The goal of this part is to evaluate how many samples are necessary to get a robust estimate of `Rc`. To do so, we will construct a learning curve.

> ***Question A22***
> - Prepare this dataset with the input feature `SM_ERA1` and output feature `Rc`.

In [14]:
# your code here


> ***Question A23***
> - Prepare an array for the learning curve sample size. The minimum size should be 10 (samples) and the maximum size the full sample size.

In [15]:
# your code here


We adopt the same strategy as the validation curve computed above

> ***Question A24***
> - Do a loop and vary the sample size. For each value $N_s$ of the sample size, you will select two subsets of size $0.8N_s$ (the train set) and $0.2N_s$ (the test set) and do a Ordinary Least Square fit. As above, construct the train and test set with a shuffled version of the original dataset. Store the value of $R^2$ for the train and test set.

In [16]:
# your code here


> ***Question A25***
> - Plot the learning curves for the train and test sets.

In [17]:
# your code here


> ***Question A26***
> - Are the learning curves monotonic? Any final comment on this learning curve?

Your text answer here:



> ***Question A27***
> - Based on the two approaches (add more features, add more samples), what is the best strategy regarding the general problematic of this notebook (prediction of RC)?

Your text answer here: 



## Question on the content of the class

> ***Question C1***
> - Explain the concept of generative methods. You can use the generative method introduced in the classification chapter as a support.

Your text answer here: 

## Bonus Questions

> ***Bonus Question B1***
> - In the Lasso regularization, the values of the coefficients go exactly to zero for high values of the regularization parameter (see Question A20). Explain why this differs from the Ridge regularization for which the values of the coefficients is never eactly zero.

Your answer:

> ***Bonus Question B2***
> - Propose and train a model to predict `Rc` that would be a natural extension of the results obtained here.

In [18]:
# your code here: